# LangChain

- LLM(대형 언어 모델)을 활용한 애플리케이션을 쉽게 만들 수 있게 해주는 Python 프레임워크
- 다양한 LLM 제공자(Google, 로컬 모델 등)를 통일된 인터페이스로 사용할 수 있다
- 프롬프트 관리, 체인 구성, 메모리, Tool 사용 등 LLM 앱에 필요한 기능을 제공한다



### LLM 앱 개발의 흐름

LLM을 활용한 서비스를 만드는 과정은 보통 다음과 같다.

1. **단순 API 호출** — LLM API를 직접 호출하여 챗봇을 만든다
2. **체인 구성** — 프롬프트 + 모델 + 파서를 연결하여 다양한 기능을 만든다
3. **메모리/Tool 추가** — 대화를 기억하고, 외부 도구(검색, DB 등)를 사용한다
4. **RAG** — 자체 문서를 검색하여 LLM에 맥락을 제공한다
5. **Agent** — LLM이 스스로 판단하여 여러 도구를 조합하고 작업을 수행한다

LangChain은 2~5단계를 쉽게 구현할 수 있게 해주는 프레임워크이다.



### LangChain 생태계

LangChain은 하나의 라이브러리가 아니라 여러 패키지로 구성된 생태계이다.

| 패키지 | 역할 | 설명 |
|--------|------|------|
| `langchain-core` | 핵심 | 기본 인터페이스, LCEL, 메시지 타입 등 |
| `langchain` | 체인/메모리 | 체인 구성, 메모리, 에이전트 등 고수준 기능 |
| `langchain-google-genai` | Google 연동 | ChatGoogleGenerativeAI 등 |
| `langchain-community` | 커뮤니티 통합 | 다양한 서드파티 Tool, 벡터 DB 등 |
| `langgraph` | Agent 프레임워크 | 상태 기반 Agent 구축 (Part 3에서 학습) |

```
langchain-core (핵심)
    ├── langchain (체인/메모리)
    ├── langchain-google-genai (Gemini 모델 연동)
    ├── langchain-community (커뮤니티)
    └── langgraph (Agent)
```



### LangChain을 사용하는 이유

Gemini API를 직접 호출해도 LLM 앱을 만들 수 있다. 하지만 기능이 복잡해질수록 직접 구현해야 할 것이 많아진다.

| 기능 | 직접 구현 | LangChain |
|------|-----------|----------|
| 프롬프트 템플릿 | 문자열 포맷팅으로 직접 관리 | `ChatPromptTemplate` |
| 모델 교체 | API별로 코드 재작성 | 한 줄 변경 (Gemini 모델 이름만 변경) |
| 대화 메모리 | 히스토리 리스트 직접 관리 | `RunnableWithMessageHistory` |
| Tool 사용 | JSON 스키마 직접 작성 + 파싱 | `@tool` 데코레이터 |
| RAG | 임베딩/검색/주입 모두 직접 구현 | `Retriever` + 체인 |
| 체인 연결 | 함수 호출 순서 직접 관리 | `|` 파이프라인 |

단순한 한 번의 호출이라면 직접 호출이 더 간단하지만, 기능이 추가될수록 LangChain의 가치가 커진다. 

## 환경 설정

필요한 패키지를 설치한다.
```bash
pip install google-genai langchain langchain-google-genai langchain-community
```




### API 키 설정

프로젝트 루트에 `.env` 파일을 만들어 API 키를 관리한다.

```dotenv
GEMINI_API_KEY=...
```

Google Gen AI SDK와 `ChatGoogleGenerativeAI`가 이 환경 변수를 자동으로 사용한다.

### 공통 모듈 및 환경 설정

In [ ]:
from dotenv import load_dotenv
from pprint import pprint
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import Runnable, RunnableLambda, RunnablePassthrough

MODEL_NAME = "gemini-3.6-flash"

load_dotenv()

---


## 직접 호출 vs LangChain 비교

In [ ]:
# Gemini Interactions API 직접 호출
from google import genai

# GEMINI_API_KEY 환경 변수를 자동으로 읽는다.
client = genai.Client()

interaction = client.interactions.create(
    model=MODEL_NAME,
    system_instruction="너는 친절한 한국어 번역가야. 다음 문장을 번역해줘: ",
    input="Hello, how are you?",
)

print(interaction.output_text)

In [ ]:
# 같은 Gemini 모델을 LangChain으로 호출

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

messages = [
    SystemMessage(content="너는 친절한 한국어 번역가야. 다음 문장을 번역해줘: "),
    HumanMessage(content="Hello, how are you?"),
]

response = llm.invoke(messages)
pprint(response.content)

직접 호출과 LangChain 호출은 같은 Gemini 모델을 사용한다. 직접 호출은 Gemini의 최신 기능을 바로 사용할 때 유용하고, LangChain은 여러 구성 요소를 연결할 때 편리하다.

1. **통일된 인터페이스** — 프롬프트, 모델, 파서를 `invoke()` 패턴으로 실행한다
2. **체인 구성** — 여러 단계의 LLM 호출을 파이프라인으로 연결할 수 있다
3. **생태계** — 메모리, Tool, RAG 등 다양한 기능이 이미 구현되어 있다

여기서 사용한 `invoke()`는 LangChain의 핵심 메서드이다. LangChain의 모든 구성 요소(LLM, 프롬프트, 파서, 체인 등)는 **Runnable**이라는 공통 인터페이스를 구현하고 있고, `invoke()`는 이 인터페이스의 기본 실행 메서드이다.

즉 `llm.invoke(messages)`는 "이 메시지 리스트를 LLM에 보내고 응답을 받아라"라는 뜻이다. 이후 배울 `prompt.invoke()`, `chain.invoke()` 등도 모두 같은 패턴이므로, **LangChain에서 무언가를 실행할 때는 `invoke()`를 쓴다**고 기억하면 된다.

---


## ChatModel

- LangChain에서 LLM을 사용하기 위한 객체
- 메시지 리스트를 입력받아 AI 응답 메시지를 반환한다

| 메시지 타입 | 역할 | 설명 |
|-------------|------|------|
| `SystemMessage` | system | LLM의 역할과 행동을 설정 |
| `HumanMessage` | user | 사용자의 입력 |
| `AIMessage` | assistant | LLM의 응답 |

In [ ]:

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

messages = [
    SystemMessage(content="너는 Python 전문가야."),
    HumanMessage(content="리스트 컴프리헨션이 뭐야?"),
]

response = llm.invoke(messages)
pprint(response)
pprint(response.content)
pprint(response.usage_metadata)
pprint(response.tool_calls)

### `response`와 `AIMessage`

`llm.invoke(messages)`의 반환값은 단순 문자열이 아니라 LangChain의 **`AIMessage` 객체**이다. `AIMessage`는 모델이 생성한 답변과 호출 관련 부가 정보를 함께 담는다.

| 속성 | 설명 |
|------|------|
| `response.content` | 모델이 생성한 실제 답변 내용 |
| `response.usage_metadata` | 입력·출력·전체 토큰 사용량 |
| `response.response_metadata` | 모델명, 종료 이유 등 제공자별 응답 정보 |
| `response.tool_calls` | 모델이 요청한 도구 호출 목록. 도구를 사용하지 않았다면 빈 리스트 |
| `response.id` | 해당 모델 응답의 식별자 |

따라서 답변 텍스트만 필요할 때는 `response.content`를 사용하고, 토큰 사용량이나 Tool 호출까지 처리할 때는 `response` 객체 전체를 사용한다.



### LLM 모델 전환

LangChain의 추상화 덕분에 같은 `llm` 변수에 다른 모델을 할당해도 동일한 `invoke()` 방식으로 호출할 수 있다.

In [ ]:

messages = [HumanMessage(content="Python의 장점 3가지를 알려줘")]

print("=== Gemini 3.6 Flash ===")
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
print(llm.invoke(messages).content)

print("\n=== Gemini 2.5 Flash Lite ===")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

print(llm.invoke(messages).content)

---


## PromptTemplate

프롬프트 안에서 바뀌는 값을 변수로 분리하면 같은 구조를 여러 입력에 재사용할 수 있다. 먼저 Python의 f-string으로 프롬프트를 만들어보자.

In [ ]:
# 프롬프트가 있음 -> 나중에 변수를 주입하고 싶음.
prompt = PromptTemplate.from_template(
    "너는 {role} 전문가야. 다음 질문에 한국어로 답해줘.\n\n질문: {question}"
)

prompt_value = prompt.invoke({
    "role": "Python",
    "question": "데코레이터가 뭐야?",
})
prompt_value = prompt.invoke({
    "role": "java",
    "question": "데코레이터가 뭐야?",
})
prompt_value = prompt.invoke({
    "role": "langchain",
    "question": "데코레이터가 뭐야?",
})

print(prompt_value.text)

# f-string
# 변수가 있음 -> 변수를 프롬프트에 넣어주고 싶음.
role = 'python'
question = "데코레이터가 뭐야?"


prompt = f"너는 {role} 전문가야. 다음 질문에 한국어로 답해줘.\n\n질문: {question}"


### f-string과 PromptTemplate 비교

f-string은 짧은 일회성 프롬프트를 만드는 데 적합하다. 프롬프트를 여러 체인에서 재사용하거나 메시지 역할을 구분할 때는 `PromptTemplate`으로 변수와 구조를 관리할 수 있다.

| 구분 | f-string | LangChain `PromptTemplate` |
|------|----------|------------------------------|
| 결과 | 일반 문자열 | LangChain이 처리하는 `PromptValue` |
| 변수 관리 | 현재 Python 변수를 직접 참조 | 필요한 `input_variables` 관리 |
| 재사용 | 문자열 생성 코드를 다시 실행 | 템플릿 객체를 여러 체인에서 재사용 |
| 메시지 역할 | 역할을 별도로 구성 | `ChatPromptTemplate`으로 역할별 관리 |
| 체인 연결 | 별도의 함수를 작성 | `prompt | llm | parser`로 연결 |
| 추적 | 최종 문자열 중심 | LangSmith에서 프롬프트 단계를 추적 |

`PromptTemplate`은 프롬프트를 재사용 가능한 실행 단위로 만들고 모델·파서와 연결하기 위해 사용한다.

### system 메시지 없는 PromptTemplate

먼저 역할 구분이 필요 없는 단일 문자열 프롬프트를 만든다. `PromptTemplate`은 완성된 문자열 형태의 `StringPromptValue`를 반환한다.

In [ ]:
prompt = PromptTemplate.from_template(
    "너는 {role} 전문가야. 다음 질문에 한국어로 답해줘.\n\n질문: {question}"
)

prompt_value = prompt.invoke({
    "role": "Python",
    "question": "데코레이터가 뭐야?",
})

print(prompt_value.text)

### system 메시지가 있는 ChatPromptTemplate

대화형 모델에서는 지시사항과 사용자 입력의 역할을 분리할 수 있다. `ChatPromptTemplate`은 `SystemMessage`, `HumanMessage` 등이 들어 있는 `ChatPromptValue`를 반환한다.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {role} 전문가야. 모든 답변은 한국어로 해줘."),
    ("human", "{question}"),
])

prompt_value = prompt.invoke({
    "role": "Python",
    "question": "데코레이터가 뭐야?",
})

print(prompt_value.messages)

두 템플릿 모두 `{role}`, `{question}`을 입력 변수로 사용하고 `invoke()`로 값을 주입한다.

- `PromptTemplate`: 역할 구분이 필요 없는 단일 문자열 프롬프트
- `ChatPromptTemplate`: system, human, ai처럼 메시지 역할을 구분하는 채팅 프롬프트

마지막 예제의 `prompt`는 이후 LCEL 예제에서 그대로 `prompt | llm | parser` 형태로 사용한다.

---


## OutputParser

- LLM의 응답을 원하는 형태로 변환하는 역할
- `llm.invoke()`의 반환값은 `AIMessage` 객체인데, `StrOutputParser`는 여기서 `content` 문자열만 깔끔하게 꺼내준다

In [ ]:

parser = StrOutputParser()

# AIMessage에서 content 문자열만 추출
result = parser.invoke(response)
print(response)
print(result)

---


## LCEL 파이프라인 (LangChain Expression Language)

- `|` 연산자로 프롬프트, 모델, 파서를 연결하여 체인을 구성한다
- 데이터가 왼쪽에서 오른쪽으로 흘러간다

```
입력 → PromptTemplate → ChatModel → OutputParser → 출력
```

In [ ]:

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {role} 전문가야."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
parser = StrOutputParser()

# LCEL 파이프라인
chain = prompt | llm | parser

result = chain.invoke({
    "role": "Python",
    "question": "리스트와 튜플의 차이가 뭐야?",
})

print(result)

In [ ]:
# chain.invoke({
#     "role": "Python",
#     "question": "리스트와 튜플의 차이가 뭐야?",
# })
# 같은 과정.
result1 = prompt.invoke({
    "role": "Python",
    "question": "리스트와 튜플의 차이가 뭐야?",
})
result2 = llm.invoke(result1)
result3 = parser.invoke(result2)

print(result3)

`chain = prompt | llm | parser`는 세 단계를 하나로 연결한 것이다.

1. `prompt` — 변수를 받아 메시지 리스트를 만든다
2. `llm` — 메시지를 받아 AI 응답을 생성한다
3. `parser` — AI 응답에서 문자열만 추출한다

이 체인에 `invoke()`를 호출하면 데이터가 순서대로 흘러가며, 최종 결과물(문자열)이 반환된다.



## Runnable

`Runnable`은 LangChain 구성 요소가 따르는 공통 실행 인터페이스다. 지금까지 사용한 프롬프트, ChatModel, 출력 파서와 이들을 연결한 체인은 모두 Runnable이다.

```text
PromptTemplate ─┐
ChatModel ──────┼─ 모두 Runnable
OutputParser ───┤
Chain ──────────┘
```

모두 같은 인터페이스를 따르기 때문에 서로 다른 종류의 객체도 `|`로 연결할 수 있다. 이때 앞 단계의 **출력 타입**이 다음 단계가 기대하는 **입력 타입**과 맞아야 한다.

In [ ]:
# prompt, llm, parser, chain은 모두 Runnable의 인스턴스다.
components = {
    "prompt": prompt,
    "llm": llm,
    "parser": parser,
    "chain": chain,
}

for name, component in components.items():
    #                                                      클래스의 이름 찾기.
    print(f"{name:>6}: {isinstance(component, Runnable)} ({type(component).__name__})")

서로 다른 클래스인 네 객체가 모두 `Runnable`이기 때문에 같은 실행 방식을 사용하고 LCEL 파이프라인으로 연결될 수 있다.

### RunnableLambda

`RunnableLambda`는 Python의 호출 가능한 객체(callable)를 Runnable로 변환한다. lambda 함수와 `def`로 정의한 일반 함수를 모두 받을 수 있다. 변환된 함수는 LCEL 파이프라인의 한 단계로 사용할 수 있으며, 주로 앞 단계의 출력을 다음 단계가 요구하는 형태로 변환한다.

In [ ]:
# def로 정의한 일반 함수를 Runnable로 변환
def to_upper_text(text: str) -> str:
    return text.upper()


to_upper = RunnableLambda(to_upper_text)

print(to_upper.invoke("hello runnable"))

# 짧은 함수는 lambda로 감쌀 수도 있다.
add_label = RunnableLambda(lambda text: f"결과: {text}")

# 두 Runnable을 | 연산자로 연결
text_chain = to_upper | add_label

print(text_chain.invoke("hello langchain"))

위 예제에서 데이터는 다음과 같이 이동한다.

```text
"hello langchain" → to_upper → "HELLO LANGCHAIN" → add_label → "결과: HELLO LANGCHAIN"
```

다음 Prompt Chaining 예제에서는 첫 번째 체인의 문자열 출력을 `{"text": ...}` 딕셔너리로 바꾸기 위해 `RunnableLambda`를 사용한다.



### Prompt Chaining 패턴

- 하나의 체인 출력을 다음 체인의 입력으로 연결하는 패턴
- LCEL의 `|` 파이프라인이 곧 Prompt Chaining이다
- 복잡한 작업을 작은 단계로 나누어 처리할 수 있다

`RunnableLambda`로 감싸는 이유: `chain1`의 출력은 `str`이지만, `chain2`의 입력은 `{"text": ...}` 딕셔너리여야 하기 때문이다. 타입 변환 어댑터 역할을 한다.

In [ ]:

# 1단계: 주제에 대한 설명 생성
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "너는 기술 블로거야. 주어진 주제에 대해 간단히 설명해줘."),
    ("human", "{topic}"),
])

# 2단계: 설명을 초보자용으로 쉽게 변환
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "너는 초보자를 위한 튜터야. 다음 설명을 {target}도 이해할 수 있게 바꿔줘."),
    ("human", "{text}"),
])

chain1 = prompt1 | llm | parser
chain2 = prompt2 | llm | parser

# chain1.invoke({'topic' : 'rest api'}) # 결과물 : str
# chain2.invoke({'text' : '뭔가의 내용'}) # 입력 : {'text' : '뭔가의 내용', 'target': '초등학생'} 의 형태
# 즉, chain1 | chain2가 불가능한 상태.
# fail_combined_chain = chain1 |  chain2

# fail_combined_chain.invoke({"topic": "REST API"})

# # RunnableLambda로 체인 연결
combined_chain = chain1 | RunnableLambda(lambda x: {"text": x, 'target' : '초등학생'}) | chain2

result = combined_chain.invoke({"topic": "REST API"})
print(result)



### RunnablePassthrough

`RunnablePassthrough`는 기존 입력을 유지하거나 기존 입력에 새 값을 추가하는 유틸리티 Runnable이다.

- `RunnablePassthrough()`: 입력을 변경하지 않고 전달
- `RunnablePassthrough.assign(key=fn)`: 입력 딕셔너리를 유지하면서 새 키와 값을 추가

프롬프트가 `question`과 `context`를 요구하는 RAG 체인에서는 호출자가 질문만 전달하고, 체인이 검색기·DB·API에서 가져온 데이터를 `context`로 추가한다.

```text
호출 입력
{"question": "Python은 누가 만들었어?"}

          ↓ 외부 검색 결과를 context로 추가

프롬프트 입력
{
    "question": "Python은 누가 만들었어?",
    "context": "검색된 문서 내용..."
}
```

이 구조를 사용하면 체인의 외부 인터페이스는 질문 입력만 받도록 단순하게 유지되고, 검색과 context 구성은 체인 내부에서 처리된다.

In [ ]:

# 사용자는 Python, Java, LangChain 중 하나를 질문으로 입력한다.
question_input = {"question": "spring"}
# question_input = {"question": "Java"}

# 외부 검색기, DB, API 대신 간단한 딕셔너리를 사용한다.
knowledge_base = {
    "Python": "Python은 귀도 반 로섬이 개발한 범용 프로그래밍 언어입니다.",
    "Java": "Java는 제임스 고슬링이 개발한 객체 지향 프로그래밍 언어입니다.",
    "LangChain": "LangChain은 언어 모델을 활용한 애플리케이션 개발을 돕는 프레임워크입니다.",
}

def retrieve_context(question):
    return knowledge_base.get(question, "")

prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 context를 참고하여 질문에 답해줘. context가 비어 있으면 모른다고 답해.\n\nContext: {context}"),
    ("human", "{question}"),
])

# 기존 입력을 유지하면서 question 값으로 조회한 데이터를 context 키로 추가
add_context = RunnablePassthrough.assign(
    context=lambda inputs: retrieve_context(inputs["question"])
)

# 체인이 외부 context를 자동으로 추가한 뒤 답변을 생성한다.
chain_with_passthrough = add_context | prompt | llm | parser
result = chain_with_passthrough.invoke(question_input)
print(result)

---


## 토큰 사용량 모니터링

- LLM API는 토큰 단위로 과금된다
- 실제 비용은 사용 모델과 입력/출력 단가에 따라 달라진다
- Gemini 호출 결과의 `usage_metadata`에서 입력·출력·전체 토큰 수를 확인할 수 있다

In [ ]:
# 토큰 사용량 직접 확인

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

response = llm.invoke([HumanMessage(content="안녕하세요")])
print(response.usage_metadata)

In [ ]:
# 체인 호출의 토큰 사용량 확인
response = llm.invoke("Python 클래스가 뭐야?")
usage = response.usage_metadata or {}

print(f"입력 토큰: {usage.get('input_tokens', 0)}")
print(f"출력 토큰: {usage.get('output_tokens', 0)}")
print(f"전체 토큰: {usage.get('total_tokens', 0)}")
print(f"\n응답: {response.content[:100]}...")

---


## LLM API 에러 핸들링

LLM API 호출은 네트워크를 통해 외부 서버에 요청을 보내는 것이기 때문에, 다양한 이유로 실패할 수 있다.

| 에러 | HTTP 코드 | 원인 | 대응 |
|------|-----------|------|------|
| Rate Limit | 429 | 짧은 시간에 너무 많은 요청 | 자동 재시도 (`max_retries`) |
| Timeout | 408/504 | 서버 응답이 너무 느림 | 타임아웃 설정 (`timeout`) |
| Server Error | 500 | Gemini 서버 장애 | 재시도 또는 fallback 모델 |
| Auth Error | 401 | API 키가 잘못됨 | `.env` 파일 확인 |

에러 처리는 보통 세 단계로 구성한다. 먼저 일시적인 오류는 재시도하고, 계속 실패하면 fallback 모델을 호출하며, 최종 실패는 `try/except`에서 처리해 사용자에게 안전한 메시지를 반환한다.

In [ ]:

# max_retries: Rate limit(429), 서버 에러(500) 시 자동 재시도 횟수
# timeout: 응답을 기다릴 최대 시간(초)
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    max_retries=3,
    timeout=30,
)

# with_fallbacks: 메인 모델이 실패하면 백업 모델로 자동 전환
llm_main = ChatGoogleGenerativeAI(model=MODEL_NAME)
llm_backup = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
llm_safe = llm_main.with_fallbacks([llm_backup])

try:
    # 재시도와 fallback까지 모두 실패하면 예외가 발생한다.
    result = llm_safe.invoke("안녕하세요")
    print(result.content)
except Exception as error:
    # 개발자 확인용: 예외 타입과 상세 메시지
    print(f"LLM 호출 실패: {error}")
    # 사용자에게 보여줄 메시지는 상세 오류와 분리한다.
    print("현재 답변을 생성할 수 없습니다. 잠시 후 다시 시도해주세요.")

`max_retries=3`으로 설정하면 에러 발생 시 자동으로 재시도한다. 대기 시간은 1초 → 2초 → 4초처럼 점점 늘어나는데, 이를 **exponential backoff**라고 한다. 무한히 빠르게 재시도하면 Rate limit이 더 심해지기 때문이다.

`with_fallbacks()`는 메인 모델이 완전히 실패했을 때 다른 모델로 자동 전환해준다. 실무에서는 비싼 모델을 메인으로, 저렴한 모델을 백업으로 두는 패턴이 흔하다.

`try/except`는 재시도와 fallback까지 모두 실패한 최종 상황을 처리한다. 예제에서는 학습을 위해 `Exception`으로 전체 오류를 잡지만, 실제 서비스에서는 인증 오류, Rate Limit, Timeout처럼 처리 방법이 다른 예외를 가능한 한 구분하는 것이 좋다. 또한 전체 오류 메시지나 API 키 같은 민감한 정보를 사용자 화면에 그대로 노출하지 않고 서버 로그에 기록해야 한다.

```text
LLM 호출 → 자동 재시도 → fallback 모델 → try/except의 최종 처리
```

실습 중 에러가 발생하면 당황하지 말고:
1. **429 에러** → 잠시 기다렸다가 다시 실행
2. **401 에러** → `.env` 파일의 API 키 확인
3. **500 에러** → Gemini 서버 문제이므로 잠시 후 재시도

---


## LLM 응답 캐싱

개발 중에는 같은 프롬프트를 반복 실행하면서 후처리 로직만 수정하는 경우가 많다. 이때 매번 API를 호출하면 비용이 낭비된다. `InMemoryCache`를 설정하면 동일한 입력에 대해 캐시된 응답을 즉시 반환한다.

In [ ]:
from time import perf_counter

set_llm_cache(InMemoryCache())

# 첫 번째 호출 — API 호출 발생
start = perf_counter()
result1 = llm.invoke("Python이 뭐야?")
print(f"첫 번째 호출: {perf_counter() - start:.2f}초")

# 두 번째 호출 — 캐시에서 즉시 반환
start = perf_counter()
result2 = llm.invoke("Python이 뭐야?")
print(f"두 번째 호출: {perf_counter() - start:.2f}초 (캐시)")

# 실습이 끝나면 캐시를 꺼두자
set_llm_cache(None)

---

## Runnable 실행 방법

모든 Runnable은 실행 목적에 따라 다음 메서드를 제공한다. 이 노트북에서는 먼저 동기 메서드의 흐름을 익힌다.

| 목적 | 동기 메서드 | 비동기 메서드 |
|------|-------------|---------------|
| 입력 하나 실행 | `invoke()` | `ainvoke()` |
| 입력 여러 개 실행 | `batch()` | `abatch()` |
| 응답 스트리밍 | `stream()` | `astream()` |

FastAPI처럼 이미 비동기로 동작하는 환경에서는 오른쪽의 비동기 메서드를 사용한다. 자세한 비동기 처리는 이 절의 마지막에서 간단히 확인한다.

### batch

여러 입력을 하나씩 `invoke()`하면 앞의 응답이 끝날 때까지 다음 요청을 보내지 못한다. `batch()`는 입력별 API 호출을 동시에 진행해 전체 대기 시간을 줄인다.

> `batch()`는 입력 목록을 Gemini에 하나의 HTTP 요청으로 보내는 기능이 아니다. 입력마다 별도의 모델 호출이 발생하며 LangChain이 그 호출들을 동시에 실행한다. 

`batch()`는 결과를 입력과 같은 순서의 리스트로 반환한다.


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 답해줘."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
chain = prompt | llm | StrOutputParser()

inputs = [
    {"question": "Python이 뭐야?"},
    {"question": "JavaScript가 뭐야?"},
    {"question": "Rust가 뭐야?"},
]

# 여러 입력을 한 번에 실행
results = chain.batch(inputs)

for result in results:
    print(result)
    print()

`batch()`는 `max_concurrency`로 동시에 실행할 요청 수를 제한할 수 있다. 요청을 너무 많이 보내면 API rate limit에 걸릴 수 있으므로 실제 서비스에서 중요한 설정이다.

In [ ]:
# 동시에 최대 2개씩만 실행
results = chain.batch(
    [
        {"question": "Go가 뭐야?"},
        {"question": "Swift가 뭐야?"},
        {"question": "Kotlin이 뭐야?"},
        {"question": "C++이 뭐야?"},
    ],
    config={"max_concurrency": 2},
)

for r in results:
    print(r)
    print()

### stream

응답 전체가 완성될 때까지 기다리지 않고 생성되는 내용을 순차적으로 표시하려면 `stream()`을 사용한다. 각 결과는 chunk 단위로 전달되므로 긴 응답도 바로 출력하기 시작할 수 있다.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "줄바꿈을 하면서 자세하게 답해줘."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
chain = prompt | llm | StrOutputParser()

# StrOutputParser가 각 응답 chunk를 문자열로 변환한다.
for chunk in chain.stream({"question": "Python의 장점 3가지를 알려줘 "}):
    print(chunk, end="", flush=True)


print()  # 줄바꿈

현재 체인은 `prompt | llm | StrOutputParser()`로 구성되어 있다. 모델은 `AIMessageChunk`를 생성하고, `StrOutputParser`가 각 chunk의 텍스트를 추출하므로 반복문에서는 문자열을 바로 사용할 수 있다.

```text
ChatGoogleGenerativeAI → AIMessageChunk → StrOutputParser → str
```

따라서 Gemini Interactions API를 직접 스트리밍할 때처럼 이벤트 종류를 확인하고 `event.delta.text`를 일일이 추출할 필요가 없다.

```python
# LangChain + StrOutputParser
for chunk in chain.stream(input_data):
    print(chunk, end="")

# Interactions API 직접 호출에서는 이벤트를 직접 처리
for event in stream:
    if event.event_type == "step.delta" and event.delta:
        if event.delta.type == "text":
            print(event.delta.text, end="")
```

chunk는 모델과 네트워크 상황에 따라 한 토큰, 여러 토큰 또는 문자 일부를 포함할 수 있다. 따라서 chunk를 항상 토큰 하나로 간주해서는 안 된다. `end=""`로 출력하면 chunk가 이어 붙으면서 자연스러운 스트리밍 효과가 난다.

### `asyncio.gather()`를 아는데 왜 `batch()`를 사용할까?

여러 비동기 호출을 동시에 실행하는 것만 목적이라면 익숙한 `asyncio.gather()`를 사용해도 된다. 실제로 LangChain의 기본 `abatch()`도 내부적으로 여러 `ainvoke()`를 병렬 실행한다.

차이는 **동시 실행의 가능 여부**가 아니라 **어떤 추상화에서 실행을 관리하는가**에 있다.

| 상황 | 권장 방식 | 이유 |
|------|-----------|------|
| 일반 동기 코드에서 같은 체인에 여러 입력 적용 | `batch()` | `await` 없이 Runnable 인터페이스로 처리 |
| 비동기 코드에서 같은 체인에 여러 입력 적용 | `abatch()` | config, callback, tracing, 동시성 설정을 LangChain 방식으로 관리 |
| 서로 다른 비동기 작업을 함께 조합 | `asyncio.gather()` | 서로 다른 coroutine을 자유롭게 구성 가능 |

따라서 `batch()`가 `gather()`보다 더 비동기적이어서 사용하는 것은 아니다. 같은 Runnable에 입력 목록을 적용할 때 LangChain의 공통 인터페이스와 실행 설정을 유지하기 위해 `batch()` 또는 `abatch()`를 사용한다.

In [ ]:
import asyncio

# 방법 1: Python의 비동기 작업으로 직접 조합
gather_results = await asyncio.gather(
    *(chain.ainvoke(item) for item in inputs)
)

# 방법 2: 같은 Runnable에 여러 입력을 적용
batch_results = await chain.abatch(
    inputs,
    config={"max_concurrency": 2},
)

for result in batch_results:
    print(result)

---


## Gemini API와 LangChain 연동 방식

이 노트북에는 서로 다른 두 가지 Gemini 호출 방식이 함께 등장한다.

| 코드 | 실제 사용하는 API | 용도 |
|------|-------------------|------|
| `client.interactions.create(...)` | Gemini **Interactions API** | Gemini의 최신 직접 호출 방식 |
| `ChatGoogleGenerativeAI(...).invoke(...)` | Gemini **generateContent API** | LangChain의 Runnable·LCEL 인터페이스 사용 |

현재 `ChatGoogleGenerativeAI`는 내부적으로 `models.generate_content()`와 `models.generate_content_stream()`을 사용한다.

또한 이 노트북에서 사용하는 Gemini 3.x 모델은 `temperature`, `top_p`, `top_k` 같은 샘플링 값을 직접 지정하지 않고 모델 기본값을 사용하는 것이 권장된다. 결과의 일관성이 필요하면 다음 방법을 우선 사용한다.

---

## 실습: 용도별 체인 만들기

아래 예상 출력을 참고하여 직접 체인을 구현해보자.

### 1. 말투 변환기 체인

같은 문장을 다양한 말투로 변환하는 체인을 만들어보자.

- 입력: `"이 기능은 다음 주까지 구현이 어려울 것 같습니다."`
- 변환할 말투: `["해적", "조선시대 임금", "츤데레 애니메이션 캐릭터"]`

예상 출력:
```
해적: 이 기능은 다음 주까지 구현하기 힘들 것 같다, 이 바다의 사나이가 말하는 거다!
조선시대 임금: 이 기능은 다음 주까지 구현하기 어려울 것이니라. 과인이 심히 염려하노라.
츤데레 애니메이션 캐릭터: 다, 다음 주까지 구현이 어렵다고?! 별로 신경 쓰이는 건 아니지만... 좀 더 시간이 필요할 뿐이야!
```

### 2. 감정분석기 체인 (few-shot)

텍스트의 감정을 분석하는 체인을 만들어보자.

- few-shot 예시를 프롬프트에 포함하여 출력 형식을 일관되게 유지한다


예상 출력:
```
입력: 이 제품 정말 최악이에요. 다시는 안 살 겁니다.
분석:
- 감정: 부정
- 강도: 강함
- 근거: "최악", "다시는 안 살 겁니다"와 같은 강한 부정 표현 사용

입력: 괜찮은 것 같아요. 가격 대비 무난합니다.
분석:
- 감정: 중립
- 강도: 약함
- 근거: "괜찮은", "무난"과 같은 중립적 표현 사용

입력: 완전 대박! 인생 최고의 구매였어요!
분석:
- 감정: 긍정
- 강도: 강함
- 근거: "완전 대박", "인생 최고"와 같은 강한 긍정 표현 사용
```

### 3. 번역기 체인

구글 번역기처럼 입력 언어와 출력 언어를 지정할 수 있는 번역 체인을 만들어보자.

- 같은 체인으로 다양한 언어 쌍을 처리할 수 있어야 한다

예상 출력:
```
한→영: The weather is really nice today. I want to go for a walk.
영→일: 今日は本当にいい天気ですね。散歩に行きたいです。
```

### 4. QA 체인 (RunnablePassthrough 활용)

`RunnablePassthrough.assign()`을 사용하여, 질문으로 딕셔너리를 검색하고 관련 배경지식을 자동으로 붙여주는 QA 체인을 만들어보자.

- 배경지식은 검색 키워드와 문서를 연결한 딕셔너리로 만든다
- 사용자는 `{"question": "..."}` 만 넘기면 질문과 관련된 문서가 `context`에 자동으로 붙어야 한다

예상 출력:
```
Q: FastAPI
A: FastAPI는 Python 기반의 고성능 웹 프레임워크이며, Starlette과 Pydantic을 기반으로 합니다.

Q: Django
A: Django는 Python 기반의 풀스택 웹 프레임워크이며, ORM과 관리자 페이지 등을 제공합니다.
```